# KOSIS OpenAPI로 인구 통계 전수부문 수집하기

이 노트북은 **국가통계포털(KOSIS) OpenAPI**에서 "인구" 주제의 통계표를 자동으로 찾아 내려받고, 표 하나당 CSV 파일 하나로 저장하는 과정을 처음부터 단계별로 설명합니다.

마지막에는 `kosis_rag_builder.py` 스크립트가 왜 지금 모양이 되었는지까지 이해할 수 있도록, 실제로 겪은 문제와 해결 과정도 함께 정리했습니다.

### 왜 이렇게 복잡한가요?

KOSIS 화면에서 표 하나를 내려받는 것은 쉽습니다. 하지만 우리가 원하는 것은 **인구 주제 아래 2,000개가 넘는 통계표 전체**입니다. 게다가 표마다 구조가 다릅니다.

- 어떤 표는 `행정구역 × 성별 × 연령`, 어떤 표는 `특성별 × 응답항목`으로 구성됩니다.
- 어떤 표는 매년, 어떤 표는 매월, 어떤 표(총조사)는 5년마다 작성됩니다.
- 같은 조사 안에도 **전수부문**(모든 사람 집계)과 **표본부문**(20% 표본 집계)이 섞여 있습니다.

그래서 사람이 하나씩 고르는 대신, 프로그램이 **폴더 구조를 끝까지 탐색하고, 각 표의 구조를 읽고, 그 구조에 맞게 요청을 나눠 받는** 방식으로 만들었습니다.

### 전체 흐름

```
① 목록 탐색 (catalog)   인구 폴더를 끝까지 내려가며 모든 통계표 찾기 → catalog.csv
        │               (표본부문 제외, 중복 표 정리)
② 구조 조사 (schema)    표마다 분류축·항목·수록시점 메타데이터 조회 → meta/*.json, table_schema.csv
        │
③ 데이터 수집 (download) 4만 셀 제한에 맞게 요청을 나눠 수치 받기 → raw/<조사명>/<기관_표>.csv
        │
④ 진행 확인 (status)    조사별 완료·실패·남은 표 확인
```

### 만들어지는 폴더

```
kosis_data/
 ├─ catalog.csv          모든 통계표 목록 (경로, 전수/표본, 선택 여부)
 ├─ table_schema.csv     표별 구조 요약 (분류축, 항목, 기간, 예상 셀 수)
 ├─ meta/                표별 메타데이터 원본 (JSON)
 ├─ raw/<조사명>/        표별 수치 데이터 (CSV) ← 최종 결과물
 ├─ parts/               내려받는 중인 조각 (중단 후 이어받기용)
 └─ failed.csv, skipped_large.csv
```

*** 셀 중 `RUN_API`가 들어간 셀은 실제 API를 호출합니다. API 키가 없어도 **예시 셀(가짜 데이터)** 은 모두 실행되므로 원리를 먼저 이해할 수 있습니다. ***

## 0. 준비

### 0-1. 필요한 패키지

```bash
pip install requests pandas python-dotenv
```

### 0-2. API 키 준비

1. [KOSIS 공유서비스](https://kosis.kr/openapi/) 회원가입
2. 활용신청 → 인증키 즉시 발급
3. 노트북과 같은 폴더에 `.env` 파일을 만들고 아래처럼 저장합니다.

```
KOSIS_API_KEY=발급받은_인증키
```

키를 코드에 직접 쓰지 않고 `.env`에 두면, 노트북을 공유해도 키가 노출되지 않습니다.

In [ ]:
import os, re, json, ast, math, time, hashlib, shutil, threading
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv(dotenv_path=Path.cwd() / ".env")
API_KEY = os.getenv("KOSIS_API_KEY")

# API 키가 있으면 실제 호출 셀을 실행하고, 없으면 예시 셀만 실행합니다.
RUN_API = bool(API_KEY)
print("API 키 확인:", "있음 → 실제 호출 셀 실행" if RUN_API else "없음 → 예시 셀만 실행")

### 0-3. 설정값

나중에 바꿀 수 있는 값은 모두 여기에 모아 둡니다.

| 설정 | 의미 |
|---|---|
| `VW_CD = "MT_ZTITLE"` | 서비스뷰: **국내통계 > 주제별** 목록을 사용 |
| `ROOT_LIST_ID = "A"` | 주제별 목록에서 **인구** 폴더의 ID |
| `CELL_LIMIT = 40_000` | 한 번 요청에 받을 수 있는 최대 셀 수 (KOSIS 제한) |
| `MIN_INTERVAL` | 호출 간 최소 간격. KOSIS는 **분당 200건** 제한이 있어 여유 있게 분당 180건으로 맞춤 |

In [2]:
BASE = "https://kosis.kr/openapi"
URL_LIST = f"{BASE}/statisticsList.do"                  # 통계목록 (폴더·표 목록)
URL_META = f"{BASE}/statisticsData.do"                  # 메타자료 (method=getMeta)
URL_DATA = f"{BASE}/Param/statisticsParameterData.do"   # 통계자료 (표를 직접 지정해 수치 조회)

VW_CD = "MT_ZTITLE"      # 국내통계 > 주제별
ROOT_LIST_ID = "A"       # 주제별 > 인구
OUT = Path("kosis_data")

CELL_LIMIT = 40_000      # 요청당 최대 셀 수
MIN_INTERVAL = 60 / 180  # 분당 180건 이하로 호출
WORKERS = 4              # 동시에 보낼 요청 수

## 1. KOSIS OpenAPI 이해하기

이 작업에 쓰는 API는 세 가지입니다.

| API | 주소 | 하는 일 | 비유 |
|---|---|---|---|
| **통계목록** | `statisticsList.do` | 폴더 안의 하위 폴더·통계표 목록 | 탐색기에서 폴더 열기 |
| **메타자료** | `statisticsData.do?method=getMeta` | 표의 분류축·항목·수록시점·주석 | 표의 설계도 |
| **통계자료** | `Param/statisticsParameterData.do` | 실제 수치 | 표의 내용 |

### 꼭 알아야 할 제한과 에러

| 코드 | 의미 | 우리 코드의 대응 |
|---|---|---|
| 30 | 조회결과 없음 | 빈 결과로 처리 (예: 자료가 없는 연도) |
| 31 | 조회결과 초과 | 요청을 4만 셀 이하로 나눠서 예방 |
| 40 | 호출 횟수 제한 (분당 200건) | 60초 기다린 뒤 재시도 |
| 20, 21 | 파라미터 누락·오류 | 실패 목록에 기록 |

### 응답 형식에서 주의할 점

`format=json`만 쓰면 KOSIS가 표준 JSON이 아닌 형식(따옴표 없는 키)을 줄 때가 있습니다. **`jsonVD=Y`** 를 붙이면 표준 JSON으로 받을 수 있습니다. 그래도 파싱이 실패하면 `ast.literal_eval`로 한 번 더 시도합니다.

### 1-1. 공통 호출 함수 `call()`

모든 API 요청은 이 함수 하나를 거칩니다. 하는 일은 네 가지입니다.

1. **호출 간격 지키기 (`_throttle`)**: 여러 스레드가 동시에 요청해도 전체 합계가 분당 180건을 넘지 않게 줄을 세웁니다.
2. **스레드별 세션**: 병렬로 요청할 때 연결을 안전하게 재사용합니다.
3. **에러코드 처리**: 30은 빈 결과, 40은 1분 대기 후 재시도, 나머지는 예외로 넘깁니다.
4. **네트워크 오류 재시도**: 일시적인 끊김은 3초, 6초, 9초 간격으로 다시 시도합니다.

In [3]:
class KosisError(Exception):
    def __init__(self, code, msg=""):
        super().__init__(f"[{code}] {msg}")
        self.code = str(code)

_tls = threading.local()          # 스레드마다 따로 쓰는 저장공간
_rate_lock = threading.Lock()     # 호출 간격 계산을 한 번에 한 스레드만 하도록
_last_call = [0.0]

def _session():
    if not hasattr(_tls, "s"):
        _tls.s = requests.Session()
    return _tls.s

def _throttle():
    """모든 스레드를 합쳐 호출 간격이 MIN_INTERVAL 이상이 되도록 대기"""
    with _rate_lock:
        wait = _last_call[0] + MIN_INTERVAL - time.monotonic()
        if wait > 0:
            time.sleep(wait)
        _last_call[0] = time.monotonic()

def call(url, params, retries=4):
    p = {"apiKey": API_KEY, "format": "json", "jsonVD": "Y", **params}
    last = None
    for attempt in range(retries):
        try:
            _throttle()
            r = _session().get(url, params=p, timeout=120)
            text = r.text.strip()
            if not text:
                return []
            try:
                data = json.loads(text)
            except json.JSONDecodeError:
                data = ast.literal_eval(text)          # 비표준 JSON 대비
            if isinstance(data, dict) and "err" in data:
                code = str(data["err"])
                if code == "30":                       # 조회결과 없음
                    return []
                if code == "40":                       # 분당 호출 제한 → 1분 대기
                    time.sleep(60)
                    continue
                raise KosisError(code, data.get("errMsg", ""))
            return data if isinstance(data, list) else [data]
        except (requests.RequestException, ValueError, SyntaxError) as e:
            last = e
            time.sleep(3 * (attempt + 1))
    raise KosisError("NET", str(last))

### 1-2. 첫 호출: 인구 폴더 열어보기

목록 API의 핵심 파라미터는 두 개입니다.

- `vwCd`: 어떤 분류 체계를 볼지 (`MT_ZTITLE` = 주제별)
- `parentListId`: **어느 폴더를 열지** (`A` = 인구)

> ⚠️ **처음에 겪은 실수**: 파라미터 이름을 `parentId`로 쓰고 값에 `"MT_ZTITLE"`을 넣었더니 원하는 목록이 오지 않았습니다. 매뉴얼상 정확한 이름은 **`parentListId`** 이고, 값은 **폴더 ID**(`A`)입니다.

응답의 각 행은 둘 중 하나입니다.

| 응답에 있는 키 | 의미 | 다음 행동 |
|---|---|---|
| `LIST_ID`, `LIST_NM` | **하위 폴더** | 그 폴더를 다시 열어 봐야 함 |
| `TBL_ID`, `TBL_NM`, `ORG_ID` | **통계표** | 수집 대상으로 기록 |

In [ ]:
if RUN_API:
    items = call(URL_LIST, {"method": "getList", "vwCd": VW_CD, "parentListId": ROOT_LIST_ID})
    df = pd.DataFrame(items)
    show = [c for c in ["LIST_ID", "LIST_NM", "ORG_ID", "TBL_ID", "TBL_NM"] if c in df.columns]
    display(df[show])
else:
    print("API 키가 없어 건너뜁니다. 아래 2장의 예시로 구조를 이해할 수 있습니다.")

## 2. 목록 트리를 끝까지 탐색하기 (순환 조회)

KOSIS 인구 폴더는 이렇게 생겼습니다.

```
인구
 ├─ 인구총조사
 │   ├─ 인구부문
 │   │   ├─ 총조사인구(2015년 이후)
 │   │   │   ├─ 전수부문 (등록센서스, 2015년 이후)
 │   │   │   │   ├─ 전수기본표
 │   │   │   │   │   └─ 📊 성 및 연령별 인구 ...
 │   │   │   │   └─ 외국인, 고령자, ...
 │   │   │   ├─ 표본(20%)부문 (2020년)
 │   │   │   └─ 표본(20%)부문 (2015년)
 │   │   └─ 세종시인구(2013년) ...
 │   ├─ 가구부문 / 시계열 연계 / 인구밀도
 ├─ 인구동향조사
 │   ├─ 📊 월.분기.연간 인구동향 ...
 │   └─ 출생 / 사망 / 혼인 / 이혼 / 다문화
 └─ 주민등록인구현황, 장래인구추계, ...
```

조사마다 **폴더 깊이가 다릅니다**. 인구총조사는 5~6단계, 인구동향조사는 2단계에서 바로 표가 나옵니다. 그래서 "몇 단계까지 내려가라"고 고정할 수 없고, **폴더가 나오면 그 폴더를 다시 여는 재귀(recursive) 방식**으로 탐색합니다.

### 재귀 탐색의 원리

```
crawl(폴더):
    목록 = 폴더 열기
    for 항목 in 목록:
        if 항목이 통계표:  기록한다 (지금까지 지나온 경로와 함께)
        if 항목이 폴더:   crawl(그 폴더)   ← 자기 자신을 다시 호출
```

**지나온 경로(path)** 를 함께 기록하는 것이 중요합니다. 경로를 보고 나중에 "이 표가 표본부문 아래에 있는지"를 판단하기 때문입니다.

### 2-1. 예시: 가짜 트리로 재귀 이해하기

API 없이 동작을 확인할 수 있도록, 작은 가짜 트리로 같은 함수를 돌려 봅니다. `list_fn`을 바꿔 끼우면 진짜 API로도, 가짜 데이터로도 같은 코드가 동작합니다.

In [5]:
def crawl(list_id, path, rows, seen, list_fn):
    """폴더(list_id)를 열고, 표는 rows에 기록하고, 하위 폴더는 다시 crawl"""
    for it in list_fn(list_id):
        if it.get("TBL_ID"):                                   # 통계표
            rows.append({
                "category": path[0] if path else "",           # 최상위 조사명 (예: 인구총조사)
                "section": path[1] if len(path) > 1 else "",   # 두 번째 단계 (예: 인구부문)
                "path": " > ".join(path),
                "depth": len(path),
                "org_id": it.get("ORG_ID"), "tbl_id": it.get("TBL_ID"), "tbl_nm": it.get("TBL_NM"),
                "stat_id": it.get("STAT_ID"), "send_de": it.get("SEND_DE"),
            })
        elif it.get("LIST_ID") and it["LIST_ID"] not in seen:  # 하위 폴더
            seen.add(it["LIST_ID"])                            # 같은 폴더를 두 번 열지 않도록
            print("  " * len(path) + f"📁 {it['LIST_NM']}")
            crawl(it["LIST_ID"], path + [it["LIST_NM"]], rows, seen, list_fn)

# ── 가짜 트리 ──
FAKE_TREE = {
    "A":   [{"LIST_ID": "A1", "LIST_NM": "인구총조사"}, {"LIST_ID": "A2", "LIST_NM": "인구동향조사"}],
    "A1":  [{"LIST_ID": "A11", "LIST_NM": "인구부문"}],
    "A11": [{"LIST_ID": "F", "LIST_NM": "전수부문 (등록센서스, 2015년 이후)"},
            {"LIST_ID": "S", "LIST_NM": "표본(20%)부문 (2020년)"},
            {"LIST_ID": "SJ", "LIST_NM": "세종시인구(2013년)"}],
    "F":   [{"ORG_ID": "101", "TBL_ID": "DT_1", "TBL_NM": "성 및 연령별 인구", "STAT_ID": "1962001"}],
    "S":   [{"ORG_ID": "101", "TBL_ID": "DT_2", "TBL_NM": "통근통학 인구(표본)", "STAT_ID": "1962001"}],
    "SJ":  [{"ORG_ID": "101", "TBL_ID": "DT_3", "TBL_NM": "세종시 연령별 인구", "STAT_ID": "1962009"}],
    "A2":  [{"ORG_ID": "101", "TBL_ID": "DT_4", "TBL_NM": "월별 출생아수", "STAT_ID": "1011001"},
            {"ORG_ID": "101", "TBL_ID": "DT_1", "TBL_NM": "성 및 연령별 인구", "STAT_ID": "1962001"}],  # 중복 표
}

rows = []
crawl("A", [], rows, set(), list_fn=lambda lid: FAKE_TREE.get(lid, []))
pd.DataFrame(rows)[["category", "path", "tbl_id", "tbl_nm"]]

📁 인구총조사
  📁 인구부문
    📁 전수부문 (등록센서스, 2015년 이후)
    📁 표본(20%)부문 (2020년)
    📁 세종시인구(2013년)
📁 인구동향조사


,category,path,tbl_id,tbl_nm
0,인구총조사,"인구총조사 > 인구부문 > 전수부문 (등록센서스, 2015년 이후)",DT_1,성 및 연령별 인구
1,인구총조사,인구총조사 > 인구부문 > 표본(20%)부문 (2020년),DT_2,통근통학 인구(표본)
2,인구총조사,인구총조사 > 인구부문 > 세종시인구(2013년),DT_3,세종시 연령별 인구
3,인구동향조사,인구동향조사,DT_4,월별 출생아수
4,인구동향조사,인구동향조사,DT_1,성 및 연령별 인구


결과를 보면 깊이가 다른 폴더(인구총조사는 3단계, 인구동향조사는 1단계)에서도 표를 빠짐없이 찾았고, 각 표가 **어떤 경로 아래에 있었는지** 함께 기록되었습니다.

`DT_1`은 두 폴더에 모두 들어 있습니다. 실제 KOSIS에서도 **같은 표가 여러 조사 폴더에 걸려 있는 경우**가 있어서, 다음 단계에서 중복을 정리합니다.

## 3. 전수부문만 고르고 중복 정리하기

### 3-1. 전수 / 표본 판별 규칙

우리 목표는 **표본부문을 제외한 데이터**입니다. 표마다 "전수/표본" 표시가 따로 있지 않으므로, **지나온 폴더 이름**으로 판단합니다.

| 경로에 이런 이름이 있으면 | 판정 | 수집 여부 |
|---|---|---|
| `표본`, `20%부문`, `샘플` | 표본 | ❌ 제외 |
| `전수`, `등록센서스` | 전수 | ✅ 수집 |
| 둘 다 없음 (예: 가구부문, 인구밀도, 인구동향조사) | 구분없음(전체집계) | ✅ 수집 |

"구분없음"도 수집하는 이유: 인구동향조사, 주민등록인구현황처럼 **애초에 표본이 없는 행정 집계 통계**가 대부분이기 때문입니다.

### 3-2. 중복 표 처리

같은 표(`org_id + tbl_id`)가 여러 폴더에 있으면 **한 번만 수집**합니다(첫 경로가 대표). 대신 다른 소속 경로는 `also_in` 컬럼에 남겨 두어, 나중에 어느 조사 이름으로 검색해도 이 표를 찾을 수 있게 합니다.

> **문제 정의**: "주택총조사"와 "세종시 특별센서스"가 결과 폴더에 없어서 누락된 줄 알았습니다. 확인해 보니 두 조사의 표는 KOSIS 트리에서 **인구총조사 폴더 안**에 들어 있어서, 모두 `raw/인구총조사/`에 저장되어 있었습니다. 인구총조사 건수 1,901 = 1,876 + 22 + 3으로 정확히 맞았습니다. **폴더 기준(category)과 작성 조사 기준(stat_id)은 다를 수 있다**는 점을 기억해 두세요.

In [6]:
SAMPLE_PATTERNS = [r"표본", r"\d+\s*%\s*부문", r"샘플"]
FULL_PATTERNS = [r"전수", r"등록센서스"]

def match_any(patterns, texts):
    return any(re.search(p, t) for p in patterns for t in texts)

def label_catalog(rows):
    cat = pd.DataFrame(rows)
    segs = cat["path"].str.split(" > ")
    cat["is_sample"] = segs.apply(lambda s: match_any(SAMPLE_PATTERNS, s))
    cat["is_full"] = segs.apply(lambda s: match_any(FULL_PATTERNS, s))
    cat["survey_type"] = cat.apply(
        lambda r: "표본" if r.is_sample else ("전수" if r.is_full else "구분없음(전체집계)"), axis=1)
    cat["selected"] = ~cat["is_sample"]

    # 중복 표: 수집은 1번만, 다른 소속 경로는 also_in 으로 보존
    cat["dup"] = cat.duplicated(["org_id", "tbl_id"])
    paths = (cat[~cat["is_sample"]].groupby(["org_id", "tbl_id"])["path"]
             .apply(lambda x: list(dict.fromkeys(x))))
    cat["also_in"] = cat.apply(
        lambda r: " || ".join(p for p in paths.get((r.org_id, r.tbl_id), []) if p != r.path), axis=1)
    cat.loc[cat["dup"], "selected"] = False
    return cat

demo_cat = label_catalog(rows)
demo_cat[["category", "path", "tbl_id", "survey_type", "dup", "selected", "also_in"]]

,category,path,tbl_id,survey_type,dup,selected,also_in
0,인구총조사,"인구총조사 > 인구부문 > 전수부문 (등록센서스, 2015년 이후)",DT_1,전수,False,True,인구동향조사
1,인구총조사,인구총조사 > 인구부문 > 표본(20%)부문 (2020년),DT_2,표본,False,False,
2,인구총조사,인구총조사 > 인구부문 > 세종시인구(2013년),DT_3,구분없음(전체집계),False,True,
3,인구동향조사,인구동향조사,DT_4,구분없음(전체집계),False,True,
4,인구동향조사,인구동향조사,DT_1,구분없음(전체집계),True,False,"인구총조사 > 인구부문 > 전수부문 (등록센서스, 2015년 이후)"


- `DT_2`(표본 20% 부문)는 `selected=False`로 제외되었습니다.
- `DT_1`은 두 번째 등장(인구동향조사 쪽)이 중복으로 제외되었지만, 대표 행의 `also_in`에 그 경로가 남았습니다.
- `DT_3`(세종시인구)는 표본 표시가 없으므로 수집 대상입니다.

### 3-3. 실제 인구 폴더 전체 탐색 → `catalog.csv`

인구 폴더 전체를 탐색하면 수백 번의 목록 호출이 일어나 몇 분 정도 걸립니다.

In [ ]:
def stage_catalog():
    rows = []
    list_fn = lambda lid: call(URL_LIST, {"method": "getList", "vwCd": VW_CD, "parentListId": lid})
    crawl(ROOT_LIST_ID, [], rows, set(), list_fn)
    cat = label_catalog(rows)
    OUT.mkdir(parents=True, exist_ok=True)
    cat.to_csv(OUT / "catalog.csv", index=False, encoding="utf-8-sig")   # 엑셀 한글 깨짐 방지: utf-8-sig
    print("\n=== 조사별 전수/표본 통계표 수 ===")
    print(cat.groupby(["category", "survey_type"]).size().unstack(fill_value=0).to_string())
    print(f"\n선택된 표: {cat['selected'].sum()} / 전체 {len(cat)}")
    return cat

if RUN_API:
    catalog = stage_catalog()

## 4. 표의 구조(메타데이터) 읽기

데이터를 받기 전에 **각 표가 어떻게 생겼는지** 알아야 한다. 그래야 몇 번에 나눠 받을지 계산할 수 있다.

메타자료 API(`method=getMeta`)는 `type` 값에 따라 다른 정보를 줍니다.

| type | 내용 | 용도 |
|---|---|---|
| `TBL` | 통계표 이름 | 확인용 |
| `ITM` | **분류축과 항목** (예: 행정구역 250개, 성별 3개, 연령 21개 / 항목: 총인구) | 요청 크기 계산, 요청 분할 |
| `PRD` | **수록주기와 시점** (예: 년, 2015~2024) | 시점 목록 만들기 |
| `CMMT` | 주석 | RAG 설명문에 활용 |
| `SOURCE` | 출처 | RAG 설명문에 활용 |

### 4-1. 분류축과 항목 구분 (`ITM`)

`ITM` 응답에는 분류 코드와 항목이 한 목록에 섞여 옵니다.

- `OBJ_ID == "ITEM"` 인 행 → **항목** (총인구, 평균연령처럼 "무엇을 셌는가")
- 나머지 → **분류**. `OBJ_ID`별로 묶고 `OBJ_ID_SN` 순서대로 정렬하면 첫 번째 분류, 두 번째 분류, … 가 됩니다.

이 순서가 나중에 데이터 요청의 `objL1`, `objL2`, … 파라미터 순서와 같습니다.

In [ ]:
def parse_itm(itm_rows):
    """ITM 메타 → (분류 레벨 리스트, 항목 리스트)"""
    levels, items = {}, []
    for r in itm_rows:
        if "_error" in r:
            continue
        if r.get("OBJ_ID") == "ITEM":
            items.append({"id": r["ITM_ID"], "nm": r.get("ITM_NM"), "unit": r.get("UNIT_NM")})
            continue
        key = r.get("OBJ_ID")
        if key not in levels:
            levels[key] = {"obj_id": key, "obj_nm": r.get("OBJ_NM"),
                           "sn": int(r.get("OBJ_ID_SN") or len(levels) + 1), "codes": []}
        levels[key]["codes"].append({"id": r["ITM_ID"], "nm": r.get("ITM_NM"), "up": r.get("UP_ITM_ID")})
    return sorted(levels.values(), key=lambda x: x["sn"]), items

# 예시: '성 및 연령별 인구' 표의 ITM 응답 일부를 흉내 낸 것 -> 가짜 데이터이지만 동작은 하도록 구조가 정의되어 있음
fake_itm = (
    [{"OBJ_ID": "A", "OBJ_NM": "행정구역별", "OBJ_ID_SN": "1", "ITM_ID": c, "ITM_NM": n}
     for c, n in [("00", "전국"), ("11", "서울특별시"), ("21", "부산광역시")]] +
    [{"OBJ_ID": "B", "OBJ_NM": "성별", "OBJ_ID_SN": "2", "ITM_ID": c, "ITM_NM": n}
     for c, n in [("0", "계"), ("1", "남자"), ("2", "여자")]] +
    [{"OBJ_ID": "ITEM", "ITM_ID": "T100", "ITM_NM": "총인구", "UNIT_NM": "명"}]
)
levels, items = parse_itm(fake_itm)
for k, l in enumerate(levels, 1):
    print(f"objL{k}: {l['obj_nm']} ({len(l['codes'])}개) → {[c['nm'] for c in l['codes']]}")
print("항목:", [(i["nm"], i["unit"]) for i in items])

objL1: 행정구역별 (3개) → ['전국', '서울특별시', '부산광역시']
objL2: 성별 (3개) → ['계', '남자', '여자']
항목: [('총인구', '명')]


### 4-2. 수록시점 읽기 (`PRD`) — 두 가지 응답 형식

> ⚠️ **실제로 겪은 문제**: 처음에는 `PRD` 응답에 시점이 하나씩 나열(`PRD_DE: 2015`, `PRD_DE: 2016`, …)된다고 가정했습니다. 그런데 결과의 예상 셀 수가 **모든 표에서 0**으로 나왔고, 다운로드가 전부 "메타 파싱 실패"로 건너뛰어졌습니다.
>
> 원인은 시점이 **범위 형식**(`STRT_PRD_DE: 2015`, `END_PRD_DE: 2024`)으로 오는 경우가 있었기 때문입니다. 그래서 **두 형식을 모두 읽고**, 범위는 주기에 맞춰 시점 목록으로 펼치도록 고쳤습니다.

| 주기 | 코드 | 시점 표기 예 |
|---|---|---|
| 년 | Y | `2024` |
| 월 | M | `202403` |
| 분기 | Q | `202401` ~ `202404` (01~04가 1~4분기) |
| 반기 | H | `202401`, `202402` |
| 2년·5년 등 다년 | F | 일단 매년으로 펼침 → 자료 없는 해는 6장에서 걸러냄 |

응답의 주기가 한글(`년`, `월`, `분기`)로 올 수도 있어서 코드로 바꾸는 표(`PRD_SE_MAP`)를 둡니다.

In [9]:
PRD_SE_MAP = {"년": "Y", "연": "Y", "반기": "H", "분기": "Q", "월": "M", "일": "D",
              "격월": "B", "부정기": "IR", "2년": "F", "3년": "F", "5년": "F", "10년": "F"}

def expand_range(se, start, end):
    """시작~종료 시점 → 시점 목록"""
    start, end = str(start).strip(), str(end).strip()
    sy, ey = int(start[:4]), int(end[:4])
    per_year = {"M": 12, "Q": 4, "H": 2}.get(se)
    if per_year is None:                                  # 년·다년·부정기 → 연 단위
        return [str(y) for y in range(sy, ey + 1)]
    s_sub, e_sub = int(start[4:6] or 1), int(end[4:6] or per_year)
    return [f"{y}{p:02d}" for y in range(sy, ey + 1) for p in range(1, per_year + 1)
            if (sy, s_sub) <= (y, p) <= (ey, e_sub)]

def parse_prd(prd_rows):
    """PRD 메타 → {주기코드: [시점, ...]}  (나열형 PRD_DE / 범위형 STRT~END 모두 처리)"""
    out = {}
    for r in prd_rows or []:
        if not isinstance(r, dict) or "_error" in r:
            continue
        se = str(r.get("PRD_SE", "")).strip()
        se = PRD_SE_MAP.get(se, se)
        if not se:
            continue
        if r.get("PRD_DE"):                                         # ① 나열형
            out.setdefault(se, []).append(str(r["PRD_DE"]))
        start, end = r.get("STRT_PRD_DE") or r.get("START_PRD_DE"), r.get("END_PRD_DE")
        if start and end:                                           # ② 범위형
            out.setdefault(se, []).extend(expand_range(se, start, end))
    return {k: sorted(set(v)) for k, v in out.items() if v}

print("범위형(년)  :", parse_prd([{"PRD_SE": "년", "STRT_PRD_DE": "2020", "END_PRD_DE": "2024"}]))
print("범위형(월)  :", parse_prd([{"PRD_SE": "월", "STRT_PRD_DE": "202311", "END_PRD_DE": "202402"}]))
print("범위형(분기):", parse_prd([{"PRD_SE": "Q", "STRT_PRD_DE": "202303", "END_PRD_DE": "202402"}]))
print("나열형      :", parse_prd([{"PRD_SE": "Y", "PRD_DE": "2020"}, {"PRD_SE": "Y", "PRD_DE": "2015"}]))

범위형(년)  : {'Y': ['2020', '2021', '2022', '2023', '2024']}
범위형(월)  : {'M': ['202311', '202312', '202401', '202402']}
범위형(분기): {'Q': ['202303', '202304', '202401', '202402']}
나열형      : {'Y': ['2015', '2020']}


### 4-3. 모든 선택 표의 메타 수집 → `meta/*.json`, `table_schema.csv`

표마다 5종(TBL, ITM, PRD, CMMT, SOURCE)의 메타를 받아 JSON으로 저장합니다. **이미 받은 표는 파일이 있으면 건너뛰므로**, 중간에 멈춰도 다시 실행하면 이어서 진행됩니다.

`table_schema.csv`에는 표마다 **예상 셀 수**를 계산해 둡니다.

```
예상 셀 수 = 항목 수 × (분류1 코드 수 × 분류2 코드 수 × …) × 시점 수
```

예를 들어 읍면동(3,500) × 성(3) × 연령(21) × 항목(1) × 10개 연도 = **약 220만 셀**이고, 한 번에 4만 셀까지만 받을 수 있으니 최소 56번 나눠 받아야 합니다. 이 숫자를 미리 보면 어떤 표가 오래 걸릴지 알 수 있습니다.

In [ ]:
def meta_path(org, tbl):
    return OUT / "meta" / f"{org}_{tbl}.json"

def get_meta(org, tbl, typ):
    try:
        return call(URL_META, {"method": "getMeta", "type": typ, "orgId": org, "tblId": tbl})
    except KosisError as e:
        return [{"_error": str(e)}]

def load_selected(surveys=None, max_tables=None):
    cat = pd.read_csv(OUT / "catalog.csv", dtype=str)
    cat = cat[cat["selected"] == "True"]
    if surveys:
        cat = cat[cat["category"].isin(surveys)]
    return cat.head(max_tables) if max_tables else cat

def stage_schema(surveys=None, max_tables=None):
    cat = load_selected(surveys, max_tables)
    (OUT / "meta").mkdir(parents=True, exist_ok=True)
    rows = []
    for i, r in enumerate(cat.itertuples(), 1):
        mp = meta_path(r.org_id, r.tbl_id)
        if mp.exists():                                                  # 캐시 재사용
            meta = json.loads(mp.read_text(encoding="utf-8"))
        else:
            meta = {t: get_meta(r.org_id, r.tbl_id, t) for t in ("TBL", "ITM", "PRD", "CMMT", "SOURCE")}
            mp.write_text(json.dumps(meta, ensure_ascii=False), encoding="utf-8")
        levels, items = parse_itm(meta["ITM"])
        prds = parse_prd(meta["PRD"])
        per_prd = max(1, len(items)) * math.prod(len(l["codes"]) for l in levels)
        rows.append({
            "category": r.category, "section": r.section, "org_id": r.org_id, "tbl_id": r.tbl_id,
            "tbl_nm": r.tbl_nm,
            "dimensions": " | ".join(f"objL{k+1}:{l['obj_nm']}({len(l['codes'])})" for k, l in enumerate(levels)),
            "n_items": len(items),
            "prd_range": ";".join(f"{k}:{v[0]}~{v[-1]}({len(v)})" for k, v in prds.items()),
            "est_cells_per_period": per_prd,
            "est_cells_total": per_prd * sum(len(v) for v in prds.values()),
        })
        if i % 50 == 0 or i == len(cat):
            print(f"[{i}/{len(cat)}] {r.tbl_nm[:30]}")
    sch = pd.DataFrame(rows)
    sch.to_csv(OUT / "table_schema.csv", index=False, encoding="utf-8-sig")
    print(sch.groupby("category")["est_cells_total"].agg(["count", "sum"]).to_string())
    return sch

if RUN_API:
    schema = stage_schema()

✅ **확인 포인트**: 출력된 `sum`(예상 셀 수 합계)이 0인 조사가 있으면 시점 메타를 읽지 못한 것입니다. `meta/` 폴더의 JSON에서 `"PRD"` 부분을 열어 형식을 확인하세요.

## 5. 4만 셀 제한에 맞게 요청 나누기

KOSIS는 한 번 요청에 **4만 셀 이하**만 돌려줍니다. 그래서 표의 크기에 따라 요청 방법을 바꿉니다.

### 경우 ① 작은 표: 여러 시점을 한 번에

시점 하나당 셀 수가 4만보다 작으면, 여러 시점을 묶어서 한 번에 받습니다.

> 예: 시도(18) × 성(3) × 항목(1) = 54셀/시점 → 4만 ÷ 54 = **740개 시점까지 한 번에** 받을 수 있음

### 경우 ② 큰 표: 시점별로, 그리고 분류 코드를 묶음으로 나눠서

시점 하나만으로도 4만 셀을 넘으면, **가장 코드가 많은 분류**를 여러 묶음으로 나눕니다. KOSIS는 코드 여러 개를 `+`로 이어 한 번에 요청할 수 있습니다 (예: `objL1=11+21+22`).

> 예: 읍면동(3,500) × 연령(20) × 항목(3) = 21만 셀/시점
> → 읍면동 코드 1개당 60셀 → 한 번에 666개까지 가능 → URL 길이를 고려해 **100개씩 묶어 35번** 요청

> 💡 **처음 버전과의 차이**: 처음에는 코드를 하나씩 따로 요청해서 같은 표가 **7,000번** 요청이 필요했습니다. `+`로 묶도록 바꾼 뒤 **35번**으로 줄었습니다.

이 계산은 재귀로 합니다. 나눈 결과가 아직 4만을 넘으면, 그다음으로 큰 분류를 또 나눕니다.

In [11]:
MAX_CODES_PER_CALL = 100     # '+'로 묶을 코드 수 상한 (URL 길이 보호)

def chunks(lst, k):
    for i in range(0, len(lst), k):
        yield lst[i:i + k]

def split_levels(levels, n_items, fixed):
    """fixed[i] = None 이면 그 분류 전체(ALL), 문자열이면 해당 코드 묶음만.
    셀 수가 한도 이하가 될 때까지 가장 큰 분류를 묶음 단위로 나눔"""
    size = lambda i: len(fixed[i].split("+")) if fixed[i] else len(levels[i]["codes"])
    est = n_items * math.prod(size(i) for i in range(len(levels)))
    free = [i for i, f in enumerate(fixed) if f is None and len(levels[i]["codes"]) > 1]
    if est <= CELL_LIMIT or not free:
        yield list(fixed)
        return
    i = max(free, key=lambda j: len(levels[j]["codes"]))       # 코드가 가장 많은 분류
    per_code = est // len(levels[i]["codes"])                   # 이 분류 코드 1개당 셀 수
    k = max(1, min(MAX_CODES_PER_CALL, CELL_LIMIT // max(1, per_code)))
    for grp in chunks([c["id"] for c in levels[i]["codes"]], k):
        f2 = list(fixed)
        f2[i] = "+".join(grp)
        yield from split_levels(levels, n_items, f2)

def plan(levels, items, periods):
    """(분류 선택, 시작시점, 종료시점) 요청 목록 만들기"""
    n_items = max(1, len(items))
    per_prd = n_items * math.prod(len(l["codes"]) for l in levels)
    if per_prd <= CELL_LIMIT:                                   # ① 작은 표
        k = max(1, CELL_LIMIT // per_prd)
        for ch in chunks(periods, k):
            yield [None] * len(levels), ch[0], ch[-1]
    else:                                                       # ② 큰 표
        for p in periods:
            for sel in split_levels(levels, n_items, [None] * len(levels)):
                yield sel, p, p

# ── 예시 ──
small = [{"obj_nm": "시도", "codes": [{"id": str(i)} for i in range(18)]},
         {"obj_nm": "성", "codes": [{"id": str(i)} for i in range(3)]}]
big = [{"obj_nm": "읍면동", "codes": [{"id": str(i)} for i in range(3500)]},
       {"obj_nm": "연령", "codes": [{"id": str(i)} for i in range(20)]}]
years = [str(y) for y in range(2015, 2025)]

p1 = list(plan(small, [1], years))
p2 = list(plan(big, [1, 2, 3], ["2024"]))
print(f"작은 표: 10개 연도 → 요청 {len(p1)}번  {p1[0][1]}~{p1[0][2]}")
print(f"큰 표  : 1개 연도 → 요청 {len(p2)}번, 첫 요청의 읍면동 코드 {len(p2[0][0][0].split('+'))}개 묶음")

작은 표: 10개 연도 → 요청 1번  2015~2024
큰 표  : 1개 연도 → 요청 35번, 첫 요청의 읍면동 코드 100개 묶음


## 6. 자료가 실제로 있는 시점만 요청하기

4-2에서 범위형 시점을 매년으로 펼쳤습니다. 그런데 인구총조사는 **5년 주기**(2010, 2015, 2020)라서, 2011~2014년처럼 자료가 없는 해가 섞입니다.

작은 표는 시점을 범위로 묶어 한 번에 요청하므로 문제가 없습니다. 하지만 **큰 표는 시점마다 수십 번씩 요청**하므로, 자료 없는 해에도 그 수십 번을 그대로 낭비하게 됩니다. 실제로 요청의 약 80%가 빈 결과였습니다.

### 해결: 아주 작은 "탐색 요청" 1번 먼저

각 분류의 **첫 번째 코드 하나씩만** 골라 전체 기간을 한 번 요청합니다. 셀 수가 매우 작아 금방 끝나고, 응답에 나온 시점(`PRD_DE`)이 곧 **자료가 실제로 있는 시점**입니다.

> 예: 5년 주기 큰 표에서 요청이 **630번 → 91번**으로 줄었습니다.

In [12]:
def fetch(org, tbl, prd_se, sel, start, end):
    """통계자료 1회 요청. sel[k]가 None이면 해당 분류 전체(ALL)"""
    params = {"method": "getList", "orgId": org, "tblId": tbl, "itmId": "ALL",
              "prdSe": prd_se, "startPrdDe": start, "endPrdDe": end}
    for k in range(8):
        params[f"objL{k+1}"] = (sel[k] or "ALL") if k < len(sel) else ""
    return call(URL_DATA, params)

def real_periods(org, tbl, prd_se, levels, items, periods, fetch_fn=fetch):
    n_items = max(1, len(items))
    per_prd = n_items * math.prod(len(l["codes"]) for l in levels)
    if per_prd <= CELL_LIMIT or len(periods) <= 1:      # 작은 표는 낭비가 없으므로 그대로
        return periods
    probe_sel = [l["codes"][0]["id"] for l in levels]   # 분류마다 첫 코드 1개
    try:
        data = fetch_fn(org, tbl, prd_se, probe_sel, periods[0], periods[-1])
    except KosisError:
        return periods
    found = sorted({str(d.get("PRD_DE")) for d in data if d.get("PRD_DE")})
    return found or periods

# ── 예시: 2000~2020을 매년으로 펼쳤지만 실제 자료는 5년 주기 ──
fake_fetch = lambda org, tbl, se, sel, s, e: [{"PRD_DE": y} for y in ("2000", "2005", "2010", "2015", "2020")
                                               if s <= y <= e]
print(real_periods("101", "X", "F", big, [1, 2, 3], [str(y) for y in range(2000, 2021)], fake_fetch))

['2000', '2005', '2010', '2015', '2020']


## 7. 내려받기: 조각 저장 + 병렬 요청

### 7-1. 왜 느렸나

진행 로그를 보면 요청 1건에 약 4초가 걸렸습니다. 데이터 양이 많아서가 아니라 **KOSIS 서버가 응답을 만드는 시간**이 대부분입니다. 그래서 다음 두 가지로 개선했습니다.

| 개선 | 효과 |
|---|---|
| **병렬 요청** (`WORKERS=4`) | 서버를 기다리는 동안 다른 요청을 동시에 보냄. 전체 호출 속도는 `_throttle`이 분당 180건 이하로 유지 |
| **조각(part) 저장** | 요청 1건의 결과를 바로 `parts/<표>/<해시>.csv`로 저장. 큰 표를 받다가 멈춰도 **받은 조각은 다시 요청하지 않음** |

조각 파일 이름은 `(주기, 분류 선택, 시작, 종료)`를 해시한 값입니다. 같은 요청이면 항상 같은 이름이 나오므로, 파일이 있는지만 보고 건너뛸 수 있습니다.

### 7-2. 저장할 컬럼

KOSIS 응답 중 다음 컬럼만 남깁니다 (영문명 등은 제외).

| 컬럼 | 의미 |
|---|---|
| `ORG_ID`, `TBL_ID`, `TBL_NM` | 기관·표 식별 |
| `C1`~`C8`, `C1_OBJ_NM`~, `C1_NM`~ | 분류 코드 / 분류 이름(예: 행정구역별) / 분류값 이름(예: 서울특별시) |
| `ITM_ID`, `ITM_NM`, `UNIT_NM` | 항목과 단위 |
| `PRD_SE`, `PRD_DE` | 주기와 시점 |
| `DT` | **수치** (문자열, 쉼표 포함 가능, `-`·`X` 같은 통계 부호도 들어올 수 있음) |
| `LST_CHN_DE` | 최종 수정일 |

이렇게 하면 결과는 **long 형식**(셀 1개 = 1행)이 됩니다. 피벗하지 않은 이 형식이 나중에 어떤 그래프로든 다시 가공하기 가장 쉽습니다.

In [13]:
KEEP_COLS = (["ORG_ID", "TBL_ID", "TBL_NM"]
             + [f"C{i}{s}" for i in range(1, 9) for s in ("", "_OBJ_NM", "_NM")]
             + ["ITM_ID", "ITM_NM", "UNIT_NM", "PRD_SE", "PRD_DE", "DT", "LST_CHN_DE"])

def safe(name):
    """폴더/파일 이름에 쓸 수 없는 문자 제거"""
    return re.sub(r'[\\/:*?"<>|\s]+', "_", str(name)).strip("_")[:80]

def fetch_part(part_dir, org, tbl, prd_se, sel, start, end):
    """요청 1건 → 조각 CSV. 이미 있으면 파일에서 읽음 (이어받기)"""
    key = hashlib.md5(f"{prd_se}|{sel}|{start}|{end}".encode()).hexdigest()[:16]
    fp = part_dir / f"{key}.csv"
    if fp.exists():
        return pd.read_csv(fp, dtype=str) if fp.stat().st_size > 3 else None
    df = pd.DataFrame(fetch(org, tbl, prd_se, sel, start, end))
    if len(df):
        df = df[[c for c in KEEP_COLS if c in df.columns]]
    part_dir.mkdir(parents=True, exist_ok=True)
    df.to_csv(fp, index=False, encoding="utf-8-sig")
    return df

### 7-3. 다운로드 본체 `stage_download`

표 하나에 대해 하는 일을 순서대로 정리하면 이렇습니다.

1. 이미 `raw/<조사>/<표>.csv`가 있으면 건너뜀 (표 단위 이어받기)
2. 메타에서 분류·항목·시점을 읽음
3. (옵션) 너무 큰 표는 건너뛰고 `skipped_large.csv`에 기록
4. 실제 자료가 있는 시점만 추림 (6장)
5. 4만 셀 이하가 되도록 요청 목록을 만듦 (5장)
6. 요청들을 병렬로 보내고, 결과는 조각으로 저장 (7-1)
7. 조각을 합쳐 중복을 제거하고 `raw/`에 저장, 조각 폴더 삭제
8. 실패한 요청은 `failed.csv`에 바로 기록

옵션은 세 가지입니다.

| 옵션 | 쓰임 |
|---|---|
| `recent=N` | 표마다 최근 N개 시점만 받기 (테스트, 또는 큰 표를 최신값만 받을 때) |
| `exclude="읍면동"` | 표 이름에 해당 단어가 있으면 제외 |
| `max_cells=5_000_000` | 예상 셀 수가 이보다 큰 표는 나중으로 미룸 |

> 💡 인구총조사의 **읍면동 단위 표**는 한 표에 4,700만 셀(요청 1,177번, 약 25분)이었습니다. 이런 표는 `max_cells`로 먼저 미뤄 두고, 필요하면 KOSIS의 **대용량 통계자료 서비스**(표 전체를 파일로 한 번에 제공)를 쓰는 것이 빠릅니다.

In [14]:
def stage_download(surveys=None, max_tables=None, recent=None, exclude=None, max_cells=None, workers=WORKERS):
    cat = load_selected(surveys, max_tables)
    if exclude:
        cat = cat[~cat["tbl_nm"].str.contains(exclude, regex=True, na=False)]
    est = {}
    if max_cells and (OUT / "table_schema.csv").exists():
        s = pd.read_csv(OUT / "table_schema.csv", dtype={"tbl_id": str})
        est = dict(zip(s["tbl_id"], s["est_cells_total"]))
    skipped = []
    for i, r in enumerate(cat.itertuples(), 1):
        out = OUT / "raw" / safe(r.category) / f"{r.org_id}_{r.tbl_id}.csv"
        if out.exists():                                                    # 1) 이미 받은 표
            continue
        if max_cells and est.get(r.tbl_id, 0) > max_cells:                  # 3) 대용량 미루기
            skipped.append({"category": r.category, "tbl_id": r.tbl_id, "tbl_nm": r.tbl_nm,
                            "est_cells_total": est[r.tbl_id]})
            continue
        mp = meta_path(r.org_id, r.tbl_id)
        if not mp.exists():
            print(f"  메타 없음 → schema 먼저 실행: {r.tbl_id}")
            continue
        meta = json.loads(mp.read_text(encoding="utf-8"))
        levels, items = parse_itm(meta["ITM"])                              # 2) 구조 읽기
        prds = parse_prd(meta["PRD"])
        if not levels or not prds:
            print(f"  ✗ 메타 파싱 실패: {r.tbl_nm} | PRD 원본: {str(meta.get('PRD'))[:150]}")
            continue

        t0 = time.time()
        part_dir = OUT / "parts" / f"{r.org_id}_{r.tbl_id}"
        tasks = []
        for prd_se, periods in prds.items():
            periods = real_periods(r.org_id, r.tbl_id, prd_se, levels, items, periods)   # 4)
            if recent:
                periods = periods[-recent:]
            tasks += [(prd_se, sel, s, e) for sel, s, e in plan(levels, items, periods)]  # 5)
        print(f"[{i}/{len(cat)}] {r.category} | {r.tbl_nm} — 요청 {len(tasks)}건", flush=True)

        frames, fails = [], []
        with ThreadPoolExecutor(max_workers=workers) as ex:                               # 6)
            futs = {ex.submit(fetch_part, part_dir, r.org_id, r.tbl_id, *t): t for t in tasks}
            for n, fu in enumerate(as_completed(futs), 1):
                prd_se, sel, s, e = futs[fu]
                try:
                    df = fu.result()
                    if df is not None and len(df):
                        frames.append(df)
                except KosisError as err:
                    fails.append({"tbl_id": r.tbl_id, "tbl_nm": r.tbl_nm, "prd": f"{s}-{e}",
                                  "sel": str(sel)[:200], "reason": str(err)})
                if len(tasks) >= 20 and n % 20 == 0:
                    print(f"    {n}/{len(tasks)} ({time.time() - t0:.0f}s)", flush=True)

        if frames:                                                                        # 7)
            df = pd.concat(frames, ignore_index=True)
            df = df[[c for c in KEEP_COLS if c in df.columns]].drop_duplicates()
            out.parent.mkdir(parents=True, exist_ok=True)
            df.to_csv(out, index=False, encoding="utf-8-sig")
            print(f"    → {len(df):,}행, {time.time() - t0:.0f}초")
            if not fails:
                shutil.rmtree(part_dir, ignore_errors=True)
        if fails:                                                                         # 8)
            fp = OUT / "failed.csv"
            pd.DataFrame(fails).to_csv(fp, mode="a", header=not fp.exists(), index=False, encoding="utf-8-sig")
    if skipped:
        pd.DataFrame(skipped).to_csv(OUT / "skipped_large.csv", index=False, encoding="utf-8-sig")
        print(f"대용량으로 미룬 표 {len(skipped)}개 → skipped_large.csv")

### 7-4. 실행하기

처음에는 **조사 하나, 표 몇 개, 최근 시점 몇 개**로 작게 시험한 뒤 전체를 실행하세요.

In [ ]:
if RUN_API:
    # ① 작게 시험: 인구총조사 표 5개, 최근 2개 시점
    stage_download(surveys=["인구총조사"], max_tables=5, recent=2)

In [ ]:
if RUN_API:
    # ② 전체 실행: 아주 큰 표(500만 셀 초과)는 미뤄 두고 나머지 먼저
    #    중간에 멈춰도 다시 실행하면 받은 표·조각은 건너뜁니다.
    stage_download(max_cells=5_000_000, workers=4)

## 8. 진행 현황 확인

조사별로 **전체 → 표본 제외 → 선택 → 메타 → 완료 → 미룸/실패 → 남은 표**를 한눈에 봅니다. 전체를 다 받았는지 확인할 때 씁니다.

In [ ]:
def stage_status():
    cat = pd.read_csv(OUT / "catalog.csv", dtype=str)
    sel = cat[cat["selected"] == "True"].copy()
    sel["meta"] = sel.apply(lambda r: meta_path(r.org_id, r.tbl_id).exists(), axis=1)
    sel["done"] = sel.apply(lambda r: (OUT / "raw" / safe(r.category) / f"{r.org_id}_{r.tbl_id}.csv").exists(),
                            axis=1)
    read_ids = lambda f: set(pd.read_csv(OUT / f, dtype=str)["tbl_id"]) if (OUT / f).exists() else set()
    sel["skipped"] = sel["tbl_id"].isin(read_ids("skipped_large.csv")) & ~sel["done"]
    sel["failed"] = sel["tbl_id"].isin(read_ids("failed.csv")) & ~sel["done"]
    allc = cat.groupby("category").agg(전체=("tbl_id", "size"),
                                       표본제외=("is_sample", lambda x: (x == "True").sum()),
                                       중복=("dup", lambda x: (x == "True").sum()))
    st = sel.groupby("category").agg(선택=("tbl_id", "size"), 메타=("meta", "sum"), 완료=("done", "sum"),
                                     미룸=("skipped", "sum"), 실패=("failed", "sum"))
    rep = allc.join(st, how="left").fillna(0).astype(int)
    rep["남음"] = rep["선택"] - rep["완료"]
    rep.loc["합계"] = rep.sum()
    return rep

if RUN_API and (OUT / "catalog.csv").exists():
    display(stage_status())

## 9. 정리: 겪은 문제와 해결

이 코드가 지금 모양이 된 이유를 시간 순서로 정리했습니다.

| # | 겪은 문제 | 원인 | 해결 |
|---|---|---|---|
| 1 | 목록이 제대로 안 옴 | 파라미터 이름을 `parentId`로 씀 | **`parentListId`** + 폴더 ID(`A`) 사용 |
| 2 | JSON 파싱 오류 | KOSIS 기본 응답이 비표준 JSON | **`jsonVD=Y`**, 실패 시 `ast.literal_eval` |
| 3 | 조사마다 폴더 깊이가 다름 | 고정 깊이 탐색 불가 | **재귀 탐색** + 지나온 경로 기록 |
| 4 | 표본부문이 섞임 | 표에 전수/표본 표시가 없음 | **경로 이름 패턴**으로 판별 |
| 5 | 같은 표가 여러 폴더에 | KOSIS가 표를 여러 곳에 연결 | 한 번만 수집, **`also_in`** 으로 소속 보존 |
| 6 | 예상 셀 수 0, 전부 파싱 실패 | 시점이 **범위형**(STRT~END)으로 옴 | 나열형·범위형 **둘 다 처리** |
| 7 | 큰 표 요청이 7,000번 | 코드를 하나씩 요청 | 코드를 **`+`로 묶어** 35번 |
| 8 | 빈 요청이 80% | 5년 주기를 매년으로 펼침 | **탐색 요청 1번**으로 실제 시점만 |
| 9 | 요청 1건에 4초, 전체가 너무 느림 | 서버 응답 대기 | **병렬 요청** + 전체 속도 제한 |
| 10 | 중간에 멈추면 처음부터 | 표 단위로만 저장 | 요청 단위 **조각 저장**으로 이어받기 |
| 11 | 읍면동 표 하나에 25분 | 4,700만 셀 | **`max_cells`로 미루기**, 필요 시 대용량 서비스 |
| 12 | 주택총조사·세종시 특별센서스가 없어 보임 | KOSIS 트리상 인구총조사 폴더 안에 있음 | 이미 수집됨. **폴더(category) ≠ 조사(stat_id)** |

## 10. 다음 단계

여기까지의 결과물(`catalog.csv`, `meta/`, `raw/`)이 **원본 데이터**입니다. 이 원본은 KOSIS 구조 그대로 보관하고, 가공은 별도 스크립트에서 합니다.

- **`kosis_views.py`**: 원본을 그래프용으로 정제(시점 정렬, 코드 순서, 합계·계층 표시)하고, 표를 그래프 단위의 "뷰"로 나눠 **BM25 + Dense 검색용 JSON**과 **데이터 좌표(`source`)** 를 만듭니다.
- 지금 노트북의 모든 기능은 **`kosis_rag_builder.py`** 에 같은 이름의 단계로 들어 있습니다.

```bash
python kosis_rag_builder.py catalog
python kosis_rag_builder.py schema
python kosis_rag_builder.py download --max-cells 5000000 --workers 4
python kosis_rag_builder.py status
```